In [2]:
import pandas as pd
import os

# لیست فایل‌ها
files = ['scale_0.001/Short/level_1.csv', 'scale_0.001/Short/level_2.csv', 'scale_0.001/Short/level_3.csv']

# خواندن و ترکیب
dataframes = []
for file in files:
    df = pd.read_csv(file)
    # استخراج عدد از نام فایل
    filename = os.path.splitext(os.path.basename(file))[0]  # 'level_1'
    level_number = filename.split('_')[1]  # '1'
    df['level_label'] = int(level_number)  # تبدیل به عدد صحیح
    dataframes.append(df)

# ترکیب تمام داده‌ها
merged_df = pd.concat(dataframes, ignore_index=True)

# ذخیره کردن
merged_df.to_csv('scale_0.001/Short/merged_data.csv', index=False)

print(f"✅ {len(files)} فایل ترکیب شدند")
print(f"📊 تعداد سطرهای نهایی: {len(merged_df)}")
print(f"📂 در 'merged_data.csv' ذخیره شد")

✅ 3 فایل ترکیب شدند
📊 تعداد سطرهای نهایی: 329726
📂 در 'merged_data.csv' ذخیره شد


## For Seperate Files - Short mode

In [ ]:
import pandas as pd
import os
import numpy as np
import random
from collections import defaultdict, deque

# List of files
files = ['scale_0.001/Short/level_1.csv', 'scale_0.001/Short/level_2.csv', 'scale_0.001/Short/level_3.csv']

# Read and add level_label
dataframes = []
for file in files:
    df = pd.read_csv(file)
    filename = os.path.splitext(os.path.basename(file))[0]
    level_number = int(filename.split('_')[1])
    df['level_label'] = level_number
    dataframes.append(df)

# Combine data
merged_df = pd.concat(dataframes, ignore_index=True)

# Create order index within each label under each level
merged_df['index_in_label_under_level'] = merged_df.groupby(['level_label', 'label']).cumcount() + 1

# Create dictionary of data lists for each (label, level_label)
print("📦 Creating data lists...")
data_dict = {}
group_stats = []

for (label, level), group in merged_df.groupby(['label', 'level_label']):
    # Sort data by index to preserve order
    sorted_group = group.sort_values('index_in_label_under_level')
    data_dict[(label, level)] = sorted_group.to_dict('records')
    group_stats.append(((label, level), len(sorted_group)))

print(f"✅ Created {len(data_dict)} lists")

# Display group statistics
print("\n📊 Group statistics:")
for (label, level), count in sorted(group_stats, key=lambda x: x[1], reverse=True):
    print(f"  ({label}, {level}): {count} samples")

# Advanced interleaving algorithm with random list selection
print("\n🎲 Starting advanced interleaving algorithm...")

# Use deque for efficiency
queues = {key: deque(data) for key, data in data_dict.items()}
all_keys = list(queues.keys())

final_data = []
total_samples = sum(len(q) for q in queues.values())
processed_samples = 0

# Configurable parameters
MIN_BATCH = 1
MAX_BATCH = 8
WEIGHTED_SELECTION = True  # Whether to use weighted selection

# Initial weights (larger lists get higher weight)
initial_weights = {key: len(data_dict[key]) for key in all_keys}
total_weight = sum(initial_weights.values())
weights = {key: initial_weights[key] / total_weight for key in all_keys}

selection_counts = {key: 0 for key in all_keys}
iteration = 0

print(f"Parameters: batch_size={MIN_BATCH}-{MAX_BATCH}, weighted_selection={WEIGHTED_SELECTION}")

while any(queues.values()):
    iteration += 1
    
    # List of non-empty keys
    active_keys = [key for key in all_keys if queues[key]]
    
    if not active_keys:
        break
    
    # **Randomly select one list from active lists**
    if WEIGHTED_SELECTION:
        # Normalized weights for active lists
        active_weights = [weights[key] for key in active_keys]
        selected_key = random.choices(active_keys, weights=active_weights, k=1)[0]
    else:
        # Uniform selection
        selected_key = random.choice(active_keys)
    
    selection_counts[selected_key] += 1
    
    # **Completely random batch size**
    # 70% small batches, 30% large batches
    if random.random() < 0.7:
        batch_size = random.randint(MIN_BATCH, min(3, MAX_BATCH))
    else:
        batch_size = random.randint(min(4, MAX_BATCH), MAX_BATCH)
    
    batch_size = min(batch_size, len(queues[selected_key]))
    
    # Take samples from the beginning of the selected list
    for _ in range(batch_size):
        if queues[selected_key]:
            final_data.append(queues[selected_key].popleft())
            processed_samples += 1
    
    # Update weight (shorter lists get lower weight)
    if WEIGHTED_SELECTION:
        remaining_ratio = len(queues[selected_key]) / max(1, len(data_dict[selected_key]))
        weights[selected_key] = max(0.1, remaining_ratio * initial_weights[selected_key] / total_weight)
    
    # Show progress
    if processed_samples % 20000 == 0 or processed_samples == total_samples:
        progress = processed_samples / total_samples * 100
        remaining = len([q for q in queues.values() if q])
        avg_batch = processed_samples / iteration
        print(f"  {processed_samples}/{total_samples} ({progress:.1f}%) - {remaining} lists - avg_batch: {avg_batch:.1f}")

print(f"\n✅ Algorithm completed!")
print(f"📊 Processed {processed_samples} samples in {iteration} iterations")
print(f"📈 Average batch size: {processed_samples/iteration:.2f}")

# Analysis of selections
print(f"\n📊 Selection analysis:")
sorted_selections = sorted(selection_counts.items(), key=lambda x: x[1], reverse=True)
print("Most selected lists:")
for i, (key, count) in enumerate(sorted_selections[:5]):
    label, level = key
    print(f"  {i+1}. ({label}, {level}): {count} times")

print("\nLeast selected lists:")
for i, (key, count) in enumerate(sorted_selections[-5:]):
    label, level = key
    print(f"  {i+1}. ({label}, {level}): {count} times")

# Convert to DataFrame and save
final_df = pd.DataFrame(final_data)
final_df.to_csv('scale_0.001/Short/CDR-MLC-Shuffle.csv', index=False)

print(f"\n📁 File saved: 'merged_data.csv' ({len(final_df)} rows)")

# Quick check of interleaving quality
print(f"\n🔍 Quick interleaving check (first 20 rows):")
sample = final_df[['label', 'level_label']].head(20)

changes = 0
prev = None
for i, row in sample.iterrows():
    current = (row['label'], row['level_label'])
    if current != prev:
        changes += 1
        prev = current

print(f"Number of changes in first 20 rows: {changes}")
print("Sample output:")
for i in range(min(10, len(final_df))):
    row = final_df.iloc[i]
    print(f"  {i+1}. {row['label']} (level {row['level_label']}) - index {row['index_in_label_under_level']}")

# Verify order preservation within each group
print(f"\n✅ Order verification:")
print("Checking if order is preserved within each (label, level_label) group...")

order_correct = True
for (label, level), original_data in data_dict.items():
    # Get this group's data from final output
    group_in_final = final_df[(final_df['label'] == label) & (final_df['level_label'] == level)]
    
    if len(group_in_final) != len(original_data):
        print(f"  ⚠️ Group ({label}, {level}): Count mismatch! Original: {len(original_data)}, Final: {len(group_in_final)}")
        order_correct = False
        continue
    
    # Check if indices are in ascending order
    indices = group_in_final['index_in_label_under_level'].tolist()
    if indices != sorted(indices):
        print(f"  ❌ Group ({label}, {level}): Order NOT preserved!")
        print(f"     Indices: {indices}")
        order_correct = False
    else:
        print(f"  ✓ Group ({label}, {level}): Order preserved ({len(indices)} samples)")

if order_correct:
    print("\n🎉 SUCCESS: Order is preserved in ALL groups!")
else:
    print("\n⚠️ WARNING: Some groups may have order issues")

# Final statistics
print(f"\n📈 Final statistics:")
print(f"Total rows: {len(final_df)}")
print(f"Unique labels: {final_df['label'].nunique()}")
print(f"Unique levels: {final_df['level_label'].nunique()}")
print(f"Unique (label, level) combinations: {len(data_dict)}")

# Distribution analysis
label_dist = final_df['label'].value_counts()
level_dist = final_df['level_label'].value_counts()

print("\nLabel distribution:")
for label, count in label_dist.items():
    percentage = count / len(final_df) * 100
    print(f"  {label}: {count} ({percentage:.1f}%)")

print("\nLevel distribution:")
for level, count in level_dist.items():
    percentage = count / len(final_df) * 100
    print(f"  Level {level}: {count} ({percentage:.1f}%)")


📦 Creating data lists...
✅ Created 15 lists

📊 Group statistics:
  (HTTP, 2): 114530 samples
  (SFTP, 2): 42437 samples
  (HTTP, 1): 39798 samples
  (HTTP, 3): 24583 samples
  (VIDEO, 2): 19967 samples
  (SFTP, 3): 15515 samples
  (SMTP, 2): 14730 samples
  (SFTP, 1): 11422 samples
  (VIDEO, 3): 9641 samples
  (SSH, 2): 9501 samples
  (VIDEO, 1): 7465 samples
  (SMTP, 3): 6778 samples
  (SMTP, 1): 6267 samples
  (SSH, 3): 3965 samples
  (SSH, 1): 3127 samples

🎲 Starting advanced interleaving algorithm...
Parameters: batch_size=1-8, weighted_selection=True
  80000/329726 (24.3%) - 13 lists - avg_batch: 3.2
  120000/329726 (36.4%) - 10 lists - avg_batch: 3.2
  140000/329726 (42.5%) - 10 lists - avg_batch: 3.2
  240000/329726 (72.8%) - 4 lists - avg_batch: 3.2
  320000/329726 (97.1%) - 1 lists - avg_batch: 3.2
  329726/329726 (100.0%) - 0 lists - avg_batch: 3.2

✅ Algorithm completed!
📊 Processed 329726 samples in 103008 iterations
📈 Average batch size: 3.20

📊 Selection analysis:
Most s

## For Single file - Long mode

In [16]:
import pandas as pd
import os
import random
from collections import deque

# List of files
files = ['scale_5/Long/CDR-MLC-notShuffle.csv']

# Read and combine all files
dataframes = [pd.read_csv(file) for file in files]
merged_df = pd.concat(dataframes, ignore_index=True)

# Create order index within each label
merged_df['index_in_label'] = merged_df.groupby('label').cumcount() + 1

# Create lists for each label
print("📦 Creating data lists...")
data_dict = {}

for label, group in merged_df.groupby('label'):
    # Sort by index to preserve order
    sorted_group = group.sort_values('index_in_label')
    data_dict[label] = sorted_group.to_dict('records')
    print(f"  {label}: {len(sorted_group)} samples")

print(f"✅ Created {len(data_dict)} lists")

# Interleaving algorithm
print("\n🎲 Starting interleaving algorithm...")

queues = {label: deque(data) for label, data in data_dict.items()}
labels = list(queues.keys())

final_data = []
total_samples = sum(len(q) for q in queues.values())
processed_samples = 0

# Parameters
MIN_BATCH = 1
MAX_BATCH = 5

selection_counts = {label: 0 for label in labels}
iteration = 0

while any(queues.values()):
    iteration += 1
    
    # Active labels (non-empty queues)
    active_labels = [label for label in labels if queues[label]]
    
    if not active_labels:
        break
    
    # Randomly select a label
    selected_label = random.choice(active_labels)
    selection_counts[selected_label] += 1
    
    # Random batch size
    batch_size = random.randint(MIN_BATCH, MAX_BATCH)
    batch_size = min(batch_size, len(queues[selected_label]))
    
    # Take samples
    for _ in range(batch_size):
        if queues[selected_label]:
            final_data.append(queues[selected_label].popleft())
            processed_samples += 1
    
    # Progress display
    if processed_samples % 10000 == 0:
        progress = processed_samples / total_samples * 100
        print(f"  Progress: {processed_samples}/{total_samples} ({progress:.1f}%)")

print(f"\n✅ Algorithm completed!")
print(f"📊 Processed {processed_samples} samples in {iteration} iterations")

# Convert to DataFrame and save
final_df = pd.DataFrame(final_data)
final_df.to_csv('scale_5/Long/CDR-MLC-Shuffle.csv', index=False)


# Verify order preservation
print(f"\n✅ Order verification:")
for label in labels:
    label_data = final_df[final_df['label'] == label]
    indices = label_data['index_in_label'].tolist()
    
    if indices == sorted(indices):
        print(f"  ✓ {label}: Order preserved ({len(indices)} samples)")
    else:
        print(f"  ❌ {label}: Order NOT preserved!")

# Quick sample
print(f"\n📋 First 15 rows:")
for i in range(min(15, len(final_df))):
    row = final_df.iloc[i]
    print(f"  {i+1}. {row['label']} - index {row['index_in_label']}")

📦 Creating data lists...
  HTTP: 99576 samples
  SFTP: 6285 samples
  SMTP: 6857 samples
  SSH: 19373 samples
  VIDEO: 84110 samples
✅ Created 5 lists

🎲 Starting interleaving algorithm...
  Progress: 10000/216201 (4.6%)
  Progress: 30000/216201 (13.9%)
  Progress: 50000/216201 (23.1%)
  Progress: 90000/216201 (41.6%)
  Progress: 120000/216201 (55.5%)
  Progress: 150000/216201 (69.4%)
  Progress: 160000/216201 (74.0%)
  Progress: 170000/216201 (78.6%)
  Progress: 180000/216201 (83.3%)
  Progress: 190000/216201 (87.9%)

✅ Algorithm completed!
📊 Processed 216201 samples in 72066 iterations

✅ Order verification:
  ✓ HTTP: Order preserved (99576 samples)
  ✓ SFTP: Order preserved (6285 samples)
  ✓ SMTP: Order preserved (6857 samples)
  ✓ SSH: Order preserved (19373 samples)
  ✓ VIDEO: Order preserved (84110 samples)

📋 First 15 rows:
  1. HTTP - index 1
  2. HTTP - index 2
  3. HTTP - index 3
  4. HTTP - index 4
  5. HTTP - index 5
  6. HTTP - index 6
  7. HTTP - index 7
  8. HTTP - inde